## Práctica, Aplicaciones con Hugging Face y Gradio

En el siguiente proyecto vamos a utilizar los modelos existentes en la libreria de Hugging Face para crear una aplicación para realizar tareas de ASR (reconocimiento automático del habla), también generaremos una interfaz de usuario mediante la libreria de Gradio.

En primer lugar cargamos nuestra API-Key de Hugging face para poder trabajar con sus modelos.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

print(f"Token loaded: {hf_token is not None}")

Token loaded: True


### Separación de las voces de la instrumental

Se realiza una separación de la pista sobre la que vamos a trabajar con el objeto de obtener mejores resultados en el reconocimiento automático del habla. Para ello hacemos uso del modelo open source Demucs y almacenamos los output de audio en ficheros utilizando la librería soundfile.

In [ ]:
import sys
import os
from demucs import pretrained
from demucs.apply import apply_model
import torchaudio
import torch

def separate_audio(input_path: str = "audio/bring-me-to-life-evanescence.mp3", output_dir: str = "stems"):
    """
    Separates an input song into stems using Demucs.
    Stems: vocals, drums, bass, other.
    """
    # Crea el directorio de pistas si no existe
    os.makedirs(output_dir, exist_ok=True)

    # Cargar modelo preentrenado Demucs
    model = pretrained.get_model('htdemucs') 
    # Seteo el modelo en modo evaluación (desactiva comportamientos de entrenamiento) 
    model.eval()

    # Cargar fichero de audio
    wav, sr = torchaudio.load(input_path)
    # Evito conversión a mono - mantengo canales originales para Demucs
    wav = wav.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
   
    model = model.to(wav.device)

    # Corro el modelo sobre el audio
    with torch.no_grad():
        estimates = apply_model(model, wav[None], split=True, overlap=0.25)[0]

    # Escribir las pistas en archivos separados
    for source, audio in zip(model.sources, estimates):
        output_path = os.path.join(output_dir, f"{source}.mp3")
        torchaudio.save(output_path, audio.cpu(), sample_rate=sr)
        print(f"Fichero guardado: {output_path}")

    print("\n Separación exitosa. Las pistas de audio se encuentran en:", os.path.abspath(output_dir))

# Execute the separation
separate_audio()

#### Tarea de ASR con Whisper en local

Uso del modelo de la librería de Hugging Face en local a través de un pipeline para realizar tareas de ASR con canciones. Nuestro objetivo en este caso es obtener la una transcripción de la letra de la canción y almacenarla en un fichero de texto con el que trabajaremos posteriormente.

In [1]:
import os
import librosa
from transformers import pipeline
import torch

# --- Configuración ---
AUDIO_FILE_PATH = os.path.join("stems", "vocals.mp3")
TRANSCRIPTION_DIR = "transcriptions"
TRANSCRIPTION_FILE_PATH = os.path.join(TRANSCRIPTION_DIR, "transcription_with_timestamps.txt")

# --- 1. Verificar Configuración y Cargar Modelo ---
print("--- Script de Transcripción ASR ---")

# Comprobar si la GPU está disponible
if torch.cuda.is_available():
    print(f"GPU disponible. Usando dispositivo: {torch.cuda.get_device_name(0)}")
    device = 0
else:
    print("GPU no encontrada. Usando CPU en su lugar. Esto será lento.")
    device = -1

# Cargar el pipeline de ASR
print("\nCargando modelo Whisper... (Esto puede tardar un momento)")
try:
    pipe = pipeline(
        "automatic-speech-recognition", 
        model="openai/whisper-large-v3",
        device=device,
        dtype=torch.float16)
    print("Modelo cargado con éxito.")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    # Salir de la celda si el modelo no se puede cargar
    exit()

# --- 2. Cargar y Transcribir Audio ---
print(f"\nProcesando archivo de audio: {AUDIO_FILE_PATH}")

# Comprobar si el archivo de audio existe
if not os.path.exists(AUDIO_FILE_PATH):
    print(f"Error: No se encontró el archivo de audio en '{AUDIO_FILE_PATH}'")
else:
    try:
        # Cargar audio usando librosa
        audio, sr = librosa.load(AUDIO_FILE_PATH, sr=16000)
        
        print("Transcribiendo audio... (Esto puede tardar dependiendo de la duración de la pista)")
        result = pipe(audio, return_timestamps=True, generate_kwargs={"task": "transcribe", "language": "en"})
        print("Transcripción completada.")

        # --- 3. Guardar y Mostrar Resultados ---
        
        # Asegurarse de que exista el directorio de salida 
        os.makedirs(TRANSCRIPTION_DIR, exist_ok=True)
        
        # Guardar la transcripción detallada con marcas de tiempo
        with open(TRANSCRIPTION_FILE_PATH, "w", encoding="utf-8") as f:
            f.write("--- Transcripción Completa ---\n")
            f.write(result['text'].strip() + "\n\n")
            f.write("--- Segmentos con Marcas de Tiempo ---\n")
            for chunk in result['chunks']:
                start_time = round(chunk['timestamp'][0], 2)
                end_time = round(chunk['timestamp'][1], 2)
                start_formatted = f"{int(start_time//60):02d}:{int(start_time%60):02d}"
                end_formatted = f"{int(end_time//60):02d}:{int(end_time%60):02d}"
                f.write(f"[{start_formatted} -> {end_formatted}] {chunk['text'].strip()}\n")
        
        print(f"\nTranscripción guardada en: {TRANSCRIPTION_FILE_PATH}")
        
        # Imprimir el texto final de la transcripción en la consola
        print("\n--- Resultado de la Transcripción ---")
        print(result['text'])

    except Exception as e:
        print(f"Ocurrió un error durante el procesamiento o la transcripción del audio: {e}")


--- Script de Transcripción ASR ---
GPU disponible. Usando dispositivo: NVIDIA GeForce RTX 3050 Laptop GPU

Cargando modelo Whisper... (Esto puede tardar un momento)


Device set to use cuda:0


Modelo cargado con éxito.

Procesando archivo de audio: stems\vocals.mp3
Transcribiendo audio... (Esto puede tardar dependiendo de la duración de la pista)
Transcripción completada.

Transcripción guardada en: transcriptions\transcription_with_timestamps.txt

--- Resultado de la Transcripción ---
 How can you see into my eyes? Like open doors Leading you down into my core Where I've become so numb And I feel that song My spirit's sleeping somewhere cold Until you find it there and leave it there Wake me up Wake me up inside Wake me up inside Call my name and save me from the dark Bid my blood to run Before I come undone Save me from the nothing I've become Now that I know what I'm worth about You can't just leave me Breathe into me Make me real Bring me to life Wake me up Wake me up inside I can't wake up Wake me up inside, call my name and save me from the dark With my blood to run, before I come undone Save me from the nothing I've become Bring me to life There's nothing inside Breat